<h1>MammoVLM &mdash; Visor diagnostico (comparador de dos columnas)</h1>
<p>Cuatro casos fijos, precomputados: dos
VinDr-Mammo (dominio de entrenamiento, intra-dominio) y dos DDSM (dominio externo,
cross-domain). Cada panel muestra dos columnas: <b>salida del modelo</b> (imagen con
mapa de saliencia Grad-CAM/IG sobre la cabeza BI-RADS del clasificador C8/exp08) y
<b>ground truth</b> (contorno o caja anotada, mas los descriptores morfologicos del
overlay/anotacion cuando existen). Un panel de concordancia campo por campo,
calculado automaticamente, hace explicito que campos predice el modelo y cuales no.</p>
<p>Casos:</p>
<ol>
  <li>VinDr BI-RADS 5 concordante, con bounding box (ancla intra-dominio)</li>
  <li>VinDr BI-RADS 1 concordante (caso normal)</li>
  <li>DDSM benigno, masa, con SHAPE y MARGINS (contraste morfologico)</li>
  <li>DDSM maligno, calcificacion, con TYPE y DISTRIBUTION, mas variante oraculo (climax)</li>
</ol>
<p>Siempre exp08 / C8, nunca exp09. Ver <code>video_brief.md</code> en el mismo
directorio para el guion de grabacion.</p>


In [1]:
## Imports de proposito general. Los imports especificos del repo (encoder,
## generador RAG, provenance, DDSM overlay, atribucion) van en la celda de
## configuracion siguiente.
import os
from pathlib import Path

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## Directorio con los casos DICOM de VinDr identificados (split TEST, ver
## scenarios/indice.csv: BI-RADS ground truth, predicho, confianza, densidad,
## malignancy score, procedencia). El comentario anterior de esta celda decia
## "20 casos" -verificado que esta desactualizado: scenarios/ tiene 68 archivos
## .dicom y 68 filas en indice.csv, cubriendo BI-RADS 1 a 5, no solo BR1/BR5.
SCENARIOS_DIR = Path("./scenarios")

## Raiz del dataset DDSM en disco. Estructura verificada en el filesystem:
##   data/6 DDSM/{benign|cancer} cases/{cat}_XX/caseYYYY/*.LJPEG.png
##     (arbol de IMAGENES)
##   data/6 DDSM/{benign|cancer} cases/{benigns|cancers}/{cat}_XX/caseYYYY/*.OVERLAY
##     (arbol de OVERLAYS, paralelo al de imagenes, NO el mismo directorio)
## La reconciliacion por ID de caso entre ambos arboles vive en
## resolve_ddsm_case() (celda de definicion de casos), no hay rutas hardcodeadas.
DDSM_ROOT = Path("./data/6 DDSM")

## Umbral congelado de VinDr para binarizar malignancy_score (P(BR4)+P(BR5)) en
## una PATHOLOGY predicha (BENIGN/MALIGNANT). Mismo valor que
## scripts/crossdomain_ddsm_auc.py: no se recalibra para DDSM a proposito, el
## umbral se fijo en el dominio de entrenamiento y se evalua tal cual cruzando
## a DDSM (ver la linea de procedencia de los casos 3 y 4).
PATHOLOGY_THRESHOLD = 0.120


In [2]:
## Interfaz real conectada al repo (inspeccion documentada en pipeline_exp08.txt).
## C8Classifier, ReportGenerator y trace_span_provenance del enunciado original NO
## existen con esos nombres. Lo que existe de verdad:
##   - src/models.py:MammoVLM               -> encoder Mammo-CLIP (EfficientNet-B5)
##                                              + cabezas BI-RADS/densidad (exp08)
##   - XAI/xai/carga_modelo.py               -> carga MammoVLM con el checkpoint
##                                              exacto de exp08 y el transform de
##                                              inferencia (CLAHE + resize 1520x912)
##   - src/rag.py:ReportRetriever            -> query morphology-blind + FAISS/PubMedBERT
##   - src/report_generator.py:ReportGenerator -> prompt + Qwen2.5-7B-Instruct
##   - XAI/xai/carga_rag.py                  -> ensambla RAG+LLM ya entrenados/indexados
##   - XAI/xai/atribucion_rag.py:calcular_atribuciones_rag -> UNICA funcion de
##     provenance que existe en el repo. Es Shapley exacto (k=3 chunks) + grounding
##     NLI (mDeBERTa-v3-mnli-xnli).
##   - XAI/xai/atribucion_clasificador.py    -> Grad-CAM + Integrated Gradients sobre
##     la cabeza BI-RADS/densidad. Recibe modelo e imagen como parametros explicitos,
##     no depende de ningun loop batch ni de un DataLoader: reusable tal cual sobre
##     una imagen suelta (verificado).
##   - src/ddsm_overlay.py                   -> parse_overlay/find_image_for_overlay/
##     draw_lesion_contour/crop_lesion_roi para el dominio DDSM (contorno reconstruido
##     desde chain code de Freeman, no bounding box).

import sys
from pathlib import Path

_TESIS_ROOT = Path.cwd()
if not (_TESIS_ROOT / "src").exists():
    _TESIS_ROOT = Path("/home/gtrujillod/Tesis")  ## fallback si el notebook no corre desde Tesis/

for _p in (str(_TESIS_ROOT / "XAI"), str(_TESIS_ROOT / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from xai import carga_modelo, carga_rag, atribucion_rag, atribucion_clasificador
import config_xai
import ddsm_overlay
import pandas as pd
import torch

## Checkpoint real del C8 (exp08), resuelto por XAI/xai/config_xai.py:
##   Tesis/outputs/experiments/exp08_ordinal_sord_qwk_descongelado/model.pt
C8_CHECKPOINT_PATH = str(carga_modelo.EXP08_MODEL_PT)


def _elegir_gpu_libre():
    ##
    ## Servidor compartido (H200 x4): device="auto" en carga_modelo/carga_rag
    ## siempre resuelve a cuda:0, y esa GPU es la que usan otros procesos del
    ## servidor -no del kernel de este notebook-. Se vio en la practica un
    ## OutOfMemoryError en Grad-CAM/IG (Integrated Gradients a 1520x912) con
    ## cuda:0 al 99.9 por ciento de uso mientras cuda:1/2/3 estaban libres.
    ## En vez de "auto", se elige en tiempo de ejecucion la GPU con mas
    ## memoria libre en este momento, y se le pasa explicitamente a las tres
    ## funciones de carga de abajo (todas aceptan un string de device, no
    ## solo "auto").
    ##
    if not torch.cuda.is_available():
        return "cpu"
    mejor_indice, mejor_libre = None, -1
    for i in range(torch.cuda.device_count()):
        try:
            libre, _total = torch.cuda.mem_get_info(i)
        except RuntimeError:
            ## Una GPU puede estar tan saturada que ni siquiera responde a la
            ## consulta de memoria libre (visto en la practica: cuda:0 al
            ## 99.9 por ciento de uso lanzo OutOfMemoryError solo al
            ## consultarla, sin llegar a cargar nada). Se descarta y se sigue
            ## con la siguiente GPU en vez de abortar toda la seleccion.
            continue
        if libre > mejor_libre:
            mejor_libre, mejor_indice = libre, i
    if mejor_indice is None:
        return "cpu"
    return f"cuda:{mejor_indice}"


_DEVICE_STR = _elegir_gpu_libre()

## Carga UNICA de todos los modelos (clasificador C8, RAG+LLM Qwen2.5-7B, NLI).
## Esto es lo que la Tabla VII reporta como 23.9 s de carga del LLM: se paga
## una sola vez aqui, no dentro de analyze_case().
classifier, _device = carga_modelo.cargar_modelo_exp08(device=_DEVICE_STR)
_transform = carga_modelo.cargar_transform_inferencia()

generator, retriever, _indexer, _llm, _tokenizer = carga_rag.cargar_pipeline_rag(device=_DEVICE_STR)
_nli_model, _nli_tokenizer, _entailment_idx = carga_rag.cargar_nli(device=_DEVICE_STR)

## Anotaciones de hallazgos de VinDr (bounding boxes de ground truth), cargadas una
## sola vez. Se usan en analyze_case() para la columna de ground truth de los casos
## VinDr (caja verde, "ground-truth lesion annotation, reference only"). Los casos
## BI-RADS 1 normalmente no tienen caja (['No Finding']) y no se dibuja nada, lo
## cual es esperado, no un error.
_FINDING_ANNOTATIONS = pd.read_csv(config_xai.FINDING_ANNOTATIONS_CSV)

print(f"Clasificador C8 (exp08) cargado en {_device}. Checkpoint: {C8_CHECKPOINT_PATH}")
print("Generador RAG (PubMedBERT + FAISS + Qwen2.5-7B-Instruct) y modelo NLI cargados.")
print(f"Anotaciones de hallazgos VinDr cargadas: {len(_FINDING_ANNOTATIONS)} filas.")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Clasificador C8 (exp08) cargado en cuda:3. Checkpoint: /home/gtrujillod/Tesis/outputs/experiments/exp08_ordinal_sord_qwk_descongelado/model.pt
Generador RAG (PubMedBERT + FAISS + Qwen2.5-7B-Instruct) y modelo NLI cargados.
Anotaciones de hallazgos VinDr cargadas: 20486 filas.


<h2>Vocabulario ingles-espanol, procedencia y veredictos (constantes auditables)</h2>
<p>Tabla EXPLICITA de mapeo de vocabulario DDSM (LESION_TYPE, SHAPE, MARGINS, TYPE,
DISTRIBUTION en ingles, tal como vienen en el overlay) a espanol. Esta tabla se usa
para (a) mostrar los descriptores del ground truth en espanol en la columna derecha,
y (b) resaltar en el reporte generado cualquier fragmento que coincida con estos
terminos -lo que hace auditable de un vistazo que ese texto viene de la literatura
recuperada (RAG) y no de una prediccion morfologica del modelo, ya que el
clasificador C8 no produce ese campo (ver panel de concordancia). Tambien viven aqui
las dos lineas de procedencia (B9, texto exacto fijado por el autor) y las cuatro
frases de veredicto (B12, texto exacto fijado por el autor).</p>


In [3]:
## Tabla de vocabulario DDSM (ingles, tal como aparece literalmente en el
## overlay) a espanol. Constante en su propia celda, no enterrada en una
## funcion, para que sea auditable de un vistazo.
##
## Cambio de rol (E5, ronda de correcciones tras C1-C5): esta tabla YA NO
## gobierna ningun veredicto de coincidencia. Antes se usaba para decidir
## "coincide"/"no coincide" comparando por subcadena contra el texto libre del
## reporte generado, y eso no es confiable: un LLM puede describir la misma
## morfologia con sinonimos o anglicismos fuera de esta lista (se observo en
## la practica: el reporte oraculo del caso 4 dice "calcificaciones coarsas",
## no "calcificaciones groseras", y por subcadena eso no matcheaba "COARSE").
## Ahora la tabla solo sirve para (a) traducir los valores del overlay al
## espanol en la columna de ground truth, y (b) extraer, para mostrarlo tal
## cual sin juzgarlo, que terminos morfologicos aparecen literalmente en el
## reporte generado (panel de concordancia, Bloque 2: yuxtaposicion sin
## veredicto entre "lo que dice el reporte" y "lo que dice la anotacion").
## No es exhaustiva de todos los codigos de DDSM ni de VinDr, cubre los que
## aparecen en los overlays/anotaciones reales de los 4 casos mas variantes
## comunes para auditoria. Desde J2 (ronda de cierre) tambien traduce
## finding_categories de VinDr (ej. SKIN_RETRACTION), no solo el overlay DDSM:
## es el mismo mecanismo de traduccion para las dos fuentes de ground truth.
VOCAB_LESION_ES = {
    ## LESION_TYPE
    "MASS": "masa",
    "CALCIFICATION": "calcificacion",
    ## SHAPE (masas)
    "ROUND": "redonda",
    "OVAL": "ovalada",
    "LOBULATED": "lobulada",
    "IRREGULAR": "irregular",
    "ARCHITECTURAL_DISTORTION": "distorsion arquitectural",
    "FOCAL_ASYMMETRIC_DENSITY": "asimetria focal",
    ## MARGINS (masas)
    "CIRCUMSCRIBED": "circunscrita",
    "MICROLOBULATED": "microlobulada",
    "OBSCURED": "oscurecida",
    "ILL_DEFINED": "margenes mal definidos",
    "SPICULATED": "espiculada",
    ## TYPE (calcificaciones)
    "PLEOMORPHIC": "pleomorfica",
    "AMORPHOUS": "amorfa",
    "FINE_LINEAR_BRANCHING": "lineal fina ramificada",
    "PUNCTATE": "puntiforme",
    "LUCENT_CENTER": "centro radiolucido",
    "COARSE": "grosera",
    ## DISTRIBUTION (calcificaciones)
    "CLUSTERED": "agrupada",
    "LINEAR": "lineal",
    "SEGMENTAL": "segmentaria",
    "REGIONAL": "regional",
    "DIFFUSELY_SCATTERED": "difusamente dispersa",
    ## finding_categories de VinDr (J2, ronda de cierre): mismo mecanismo de
    ## traduccion que el resto de la tabla, aplicado al vocabulario de
    ## finding_annotations.csv en vez del overlay DDSM. "MASS" ya esta
    ## arriba y se reusa tal cual para ambos dominios.
    "SUSPICIOUS_CALCIFICATION": "calcificacion sospechosa",
    "ASYMMETRY": "asimetria",
    "FOCAL_ASYMMETRY": "asimetria focal",
    "SKIN_THICKENING": "engrosamiento cutaneo",
    "SKIN_RETRACTION": "retraccion cutanea",
    "NIPPLE_RETRACTION": "retraccion del pezon",
    "SUSPICIOUS_LYMPH_NODE": "ganglio linfatico sospechoso",
    "NO_FINDING": "sin hallazgo",
}

## Lineas de procedencia (B9). Texto exacto, no se abrevia ni se reformula.
PROVENANCE_DDSM = (
    "Seleccion sobre DDSM (N=3669) con el umbral congelado de VinDr (0.120): "
    "clasificacion correcta en 1695/1805 malignos (93.9 por ciento) y en "
    "152/1864 benignos (8.2 por ciento). La sensibilidad alta refleja un sesgo "
    "del umbral hacia la categoria maligna, no capacidad discriminativa "
    "(AUC cross-domain 0.5438)."
)

PROVENANCE_VINDR = (
    "Caso del split TEST de VinDr-Mammo. BI-RADS predicho concordante con la "
    "anotacion de referencia. Dominio de entrenamiento."
)

## Procedencia especifica del caso ancla (caso 1, G2 de la ronda de cierre).
## Distinta de PROVENANCE_VINDR (que sigue siendo la del caso 2): el caso 1 se
## eligio, entre los concordantes, por concentracion de la saliencia sobre la
## lesion anotada (ver outputs/analisis_interno/seleccion_caso_ancla.csv, que
## NO se publica: la seleccion se declara en pantalla, el numero no).
PROVENANCE_VINDR_ANCLA = (
    "Caso del split TEST de VinDr-Mammo. BI-RADS predicho concordante con la "
    "anotacion de referencia. Dominio de entrenamiento. Caso seleccionado entre "
    "los concordantes por concentracion de la saliencia sobre la lesion "
    "anotada: demuestra que el encoder puede localizar, no que lo haga de forma "
    "uniforme."
)

## Frases de veredicto (B12). Texto exacto fijado por el autor, una por caso.
VERDICT_TEXTS = {
    1: (
        "El encoder concentra la saliencia sobre la masa anotada y la "
        "clasificacion BI-RADS es correcta. Aun asi, el reporte describe "
        "calcificaciones coarsas y heterogeneas cuando la anotacion indica una masa "
        "con retraccion cutanea. El texto no proviene de la imagen ni cuando la "
        "clasificacion acierta."
    ),
    2: (
        "No hay hallazgo anotado. El generador produce igualmente un reporte "
        "completo y estructurado. La forma del texto no depende de que exista algo "
        "que describir."
    ),
    3: (
        "En dominio externo la patologia binaria coincide, la categoria BI-RADS no. "
        "El reporte no describe forma ni margenes, pese a que la anotacion los "
        "especifica."
    ),
    4: (
        "Con los escalares predichos, el reporte niega hallazgos sospechosos sobre "
        "un cancer confirmado por biopsia. Con los escalares del ground truth, "
        "describe calcificaciones coarsas y heterogeneas, trazables a la pagina 94 "
        "de la referencia BI-RADS recuperada, cuando la anotacion indica "
        "pleomorficas de distribucion lineal. Corregir los escalares no corrige la "
        "morfologia."
    ),
}

## Terminos de EXTRACCION para el panel de concordancia (ya no de veredicto,
## ver comentario de VOCAB_LESION_ES arriba). Bug de alcance corregido (L3,
## ronda de correcciones): antes MORPHOLOGY_TERMS_ES se derivaba por
## EXCLUSION de VOCAB_LESION_ES ("todo lo que no sea MASS/CALCIFICATION"), y
## cuando J2 agrego a esa misma tabla el vocabulario de finding_categories de
## VinDr (SKIN_RETRACTION, NO_FINDING, ASYMMETRY, etc., que NO son morfologia),
## esos terminos empezaron a colarse como "morfologia" extraible del reporte.
## VOCAB_LESION_ES sigue siendo el diccionario de TRADUCCION de ground truth
## (DDSM overlay + finding_categories de VinDr); estas dos listas de abajo son
## EXPLICITAS y escritas a mano, para que el alcance de lo que se busca en el
## texto libre del reporte sea auditable de un vistazo, sin depender de que
## alguien recuerde excluir las claves nuevas cada vez que la tabla de
## traduccion crezca.
LESION_TYPE_TERMS_ES = ["masa", "calcificacion"]

MORPHOLOGY_TERMS_ES = [
    ## SHAPE (masas, DDSM)
    "redonda", "ovalada", "lobulada", "irregular",
    "distorsion arquitectural", "asimetria focal",
    ## MARGINS (masas, DDSM)
    "circunscrita", "microlobulada", "oscurecida",
    "margenes mal definidos", "espiculada",
    ## TYPE (calcificaciones, DDSM)
    "pleomorfica", "amorfa", "lineal fina ramificada",
    "puntiforme", "centro radiolucido", "grosera",
    ## DISTRIBUTION (calcificaciones, DDSM)
    "agrupada", "lineal", "segmentaria", "regional", "difusamente dispersa",
]


<h2>Definicion de los 4 casos fijos y reconciliacion DDSM por ID de caso</h2>
<p>Los 4 casos quedaron fijados tras el PASO A de verificacion (ver conversacion):
dos VinDr (rutas directas a <code>scenarios/</code>) y dos DDSM (identificados por
<code>case_id</code> + vista, no por ruta). Las imagenes y los overlays de DDSM
viven en arboles paralelos en el filesystem
(<code>cancer_XX/caseYYYY/*.LJPEG.png</code> frente a
<code>cancers/cancer_XX/caseYYYY/*.OVERLAY</code>), asi que la reconciliacion se
implementa como funcion (<code>resolve_ddsm_case</code>), no como ruta hardcodeada.</p>


In [4]:
def find_ddsm_overlay_file(case_id, view):
    ##
    ## Busca el archivo .OVERLAY de un caso+vista dados, recorriendo el arbol
    ## de overlays (data/6 DDSM/.../{benigns|cancers}/.../caseYYYY/*.OVERLAY).
    ## No hay una tabla precomputada de rutas: se busca por ID de caso en el
    ## filesystem cada vez, tal como pidio el autor.
    ##
    for root, _dirs, files in os.walk(DDSM_ROOT):
        if os.path.basename(root) != case_id:
            continue
        for f in files:
            if f.upper().endswith(".OVERLAY") and view in f.split("."):
                return Path(root) / f
    return None


def resolve_ddsm_case(case_id, view):
    ##
    ## Reconciliacion por ID de caso entre los arboles paralelos de imagen y
    ## overlay (ver docstring de src/ddsm_overlay.py). No hay rutas
    ## hardcodeadas: se busca el overlay por case_id+view con
    ## find_ddsm_overlay_file() y se resuelve la imagen correspondiente con
    ## ddsm_overlay.find_image_for_overlay(), ya existente en el repo.
    ##
    overlay_path = find_ddsm_overlay_file(case_id, view)
    if overlay_path is None:
        raise FileNotFoundError(
            f"Overlay no encontrado para {case_id} vista {view} en {DDSM_ROOT}"
        )
    image_path = ddsm_overlay.find_image_for_overlay(overlay_path)
    if image_path is None:
        raise FileNotFoundError(f"Imagen no encontrada para overlay {overlay_path}")
    return Path(image_path), Path(overlay_path)


## Indice de referencia de VinDr (BI-RADS ground truth, predicho, grupo,
## confianza, densidad, malignancy score). Cargado una sola vez.
_SCENARIOS_INDEX = pd.read_csv(SCENARIOS_DIR / "indice.csv").set_index("archivo")

## Los 4 casos fijos confirmados por el autor tras el PASO A. "numero" identifica
## el boton/panel; "procedencia" es el texto exacto de B9; "oraculo" marca el
## unico caso (4) con variante de escalares de ground truth (B7).
CASE_SOURCES = {
    1: {
        "numero": 1,
        "kind": "vindr",
        ## G1 (ronda de cierre): sustituido maligno_case011 por
        ## maligno_case019. case011 quedo en la posicion 11 de 12 en la
        ## concentracion de saliencia sobre la lesion anotada (ver
        ## outputs/analisis_interno/seleccion_caso_ancla.csv); case019 tiene
        ## el mayor enriquecimiento sobre el nivel esperable por azar (14.2x,
        ## caja pequena, 5.5 por ciento del area de la imagen). BI-RADS 5/5
        ## concordante, confianza 0.557, malignancy_score 0.868, bbox
        ## recuperable.
        "path": SCENARIOS_DIR / "maligno_case019_BR5_mal0.87.dicom",
        "categoria": "maligno",
        "origen": "VinDr-Mammo",
        "procedencia": PROVENANCE_VINDR_ANCLA,
    },
    2: {
        "numero": 2,
        "kind": "vindr",
        "path": SCENARIOS_DIR / "benigno_case001_BR1_mal0.05.dicom",
        "categoria": "normal",
        "origen": "VinDr-Mammo",
        "procedencia": PROVENANCE_VINDR,
    },
    3: {
        "numero": 3,
        "kind": "ddsm",
        "case_id": "case3391",
        "view": "RIGHT_CC",
        "categoria": "benigno",
        "origen": "DDSM",
        "procedencia": PROVENANCE_DDSM,
    },
    4: {
        "numero": 4,
        "kind": "ddsm",
        "case_id": "case1081",
        "view": "RIGHT_CC",
        "categoria": "maligno",
        "origen": "DDSM",
        "procedencia": PROVENANCE_DDSM,
        "oraculo": True,
    },
}

## Cache de texto (C4). Solo el texto de los reportes (predicho y oraculo,
## mas query/chunks/provenance) se guarda aqui: es la parte cara (LLM de 7B +
## Shapley de 8 subconjuntos) y NO determinista (verificado: 3 corridas de
## generator.generate() con el mismo prediction_dict dieron 2 textos
## distintos), asi que recalcularla en cada apertura del notebook arriesgaria
## grabar un texto distinto al ya auditado. La imagen, la saliencia (Grad-CAM
## + IG) y el contorno de ground truth NO se cachean aqui: se recalculan en
## vivo en cada corrida porque son baratos y deterministas.
import json

CASE_CACHE_PATH = Path("./outputs/demo_cache/case_results_cache.json")

## Poner en True una sola vez para forzar regeneracion del texto (por ejemplo
## si se reentrena exp08 o se reindexa el RAG), luego volver a False. Con
## False, si el archivo de cache existe se reusa tal cual.
FORCE_REGENERATE_TEXT = False


def _load_text_cache():
    if CASE_CACHE_PATH.exists():
        return json.loads(CASE_CACHE_PATH.read_text(encoding="utf-8"))
    return {}


def _save_text_cache(cache):
    CASE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    CASE_CACHE_PATH.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")


<h2>Funcion de analisis: caso adentro, resultados afuera</h2>
<p>Corre el pipeline C8/exp08 completo sobre un caso (VinDr o DDSM) y devuelve un
diccionario con todo lo que la interfaz necesita mostrar: clasificacion, saliencia
(Grad-CAM + Integrated Gradients), ground truth de la columna derecha, panel de
concordancia campo por campo, reporte generado y evidencia de auditoria. Para el
caso 4 (unico con variante oraculo) tambien incluye un segundo reporte generado con
los escalares del ground truth. Es el unico lugar donde se conectan las llamadas
reales al repo; el resto del notebook solo consume lo que esta funcion devuelve.</p>


In [5]:
import re
import ast
import torch
import pydicom

from report_generator import DENSITY_DESCRIPTIONS_ES


def _parse_finding_categories(raw):
    ##
    ## finding_categories en finding_annotations.csv viene como texto de una
    ## lista Python literal, ej "['Mass']". ast.literal_eval lo parsea sin
    ## eval() arbitrario.
    ##
    if not raw:
        return []
    try:
        parsed = ast.literal_eval(raw)
    except Exception:
        return [str(raw)]
    if isinstance(parsed, list):
        return [str(c) for c in parsed]
    return [str(parsed)]


def _lookup_vindr_ground_truth(dicom_path):
    ##
    ## Ground truth de un caso VinDr: bounding boxes, BI-RADS ground truth y
    ## grupo (scenarios/indice.csv), mas -para el panel de concordancia-
    ## finding_categories (Mass/Calcification/etc.) y breast_density de
    ## finding_annotations.csv. VinDr NO trae columnas SHAPE/MARGINS/TYPE/
    ## DISTRIBUTION por hallazgo (no existen en el CSV): por eso la morfologia
    ## de ground truth queda vacia para VinDr, no por un descuido del codigo.
    ##
    ## gt_box_labels_en (O2, ronda de correcciones): cada fila de
    ## finding_annotations.csv es UN hallazgo con SU PROPIA caja y SU PROPIA
    ## finding_categories (verificado en el caso 1: fila 1 = caja de Skin
    ## Retraction, fila 2 = caja de Mass, cada una con su propio bbox). Por
    ## eso el rotulo por caja se arma emparejando categoria y caja EN LA
    ## MISMA ITERACION de fila, no como listas separadas -a diferencia de
    ## gt_lesion_types_en (usado en el panel de concordancia), que sigue
    ## siendo la lista deduplicada de categorias del caso completo, sin
    ## asociacion a una caja especifica.
    ##
    try:
        ds = pydicom.dcmread(str(dicom_path), stop_before_pixels=True)
        image_id = str(ds.SOPInstanceUID)
    except Exception:
        image_id = None

    boxes, ref_size = [], None
    gt_box_labels_en = []
    gt_density_letter = None
    gt_lesion_types_en = []
    if image_id is not None:
        rows = _FINDING_ANNOTATIONS[_FINDING_ANNOTATIONS["image_id"] == image_id]
        if not rows.empty:
            ref_size = (int(rows.iloc[0]["height"]), int(rows.iloc[0]["width"]))
            first_density = str(rows.iloc[0].get("breast_density", "") or "")
            if first_density.upper().startswith("DENSITY"):
                gt_density_letter = first_density.strip().split()[-1].upper()
            for _, r in rows.iterrows():
                row_cats = [
                    cat.strip().upper().replace(" ", "_")
                    for cat in _parse_finding_categories(r.get("finding_categories", ""))
                ]
                if pd.notna(r["xmin"]):
                    boxes.append(
                        (float(r["xmin"]), float(r["ymin"]), float(r["xmax"]), float(r["ymax"]))
                    )
                    gt_box_labels_en.append(row_cats)
                for cat_en in row_cats:
                    if cat_en and cat_en not in gt_lesion_types_en:
                        gt_lesion_types_en.append(cat_en)

    idx_row = _SCENARIOS_INDEX.loc[Path(dicom_path).name]
    gt_birads = int(idx_row["birads_ground_truth"])
    gt_pathology = "MALIGNANT" if idx_row["grupo"] == "maligno" else "BENIGN"

    return {
        "gt_boxes": boxes,
        "gt_box_labels_en": gt_box_labels_en,
        "gt_box_ref_size": ref_size,
        "gt_birads": gt_birads,
        "gt_pathology": gt_pathology,
        "n_findings": len(boxes),
        "lesions": [],  ## VinDr no trae SHAPE/MARGINS/TYPE/DISTRIBUTION por hallazgo
        "gt_density_letter": gt_density_letter,
        "gt_lesion_types_en": gt_lesion_types_en,
    }


def _lookup_ddsm_ground_truth(overlay_path):
    ##
    ## Ground truth de un caso DDSM: parsea el overlay completo con
    ## ddsm_overlay.parse_overlay (ya existente en el repo) y expone TODAS las
    ## anomalias, no solo la primera -soporte para N hallazgos (B6): 236 de
    ## 3671 overlays en disco tienen 2 o mas anomalias, verificado en el PASO A.
    ## DDSM no anota densidad por caso (verificado: el formato OVERLAY no
    ## tiene ningun campo DENSITY), por eso gt_density_letter siempre es None.
    ##
    overlay_data = ddsm_overlay.parse_overlay(overlay_path)
    lesions = []
    for les in overlay_data.lesions:
        if les.lesion_type == "MASS":
            morfologia_en = {"SHAPE": les.mass_shape, "MARGINS": les.mass_margins}
        elif les.lesion_type == "CALCIFICATION":
            morfologia_en = {"TYPE": les.calc_type, "DISTRIBUTION": les.calc_distribution}
        else:
            morfologia_en = {}
        lesions.append({
            "lesion_type": les.lesion_type,
            "morfologia_en": morfologia_en,
            "assessment": les.assessment,
            "subtlety": les.subtlety,
            "pathology": les.pathology,
            "contours": les.contours,
        })

    primary = lesions[0] if lesions else None

    return {
        "overlay_data": overlay_data,
        "lesions": lesions,
        "gt_birads": primary["assessment"] if primary else None,
        "gt_pathology": primary["pathology"] if primary else "UNKNOWN",
        "n_findings": overlay_data.total_abnormalities,
        "gt_density_letter": None,
    }


def _build_oracle_prediction_dict(base_pred, oracle_birads_idx, oracle_malignancy):
    ##
    ## Envuelve carga_rag.prediccion_a_dict_generador() sustituyendo el BI-RADS
    ## y el malignancy_score predichos por los del ground truth (variante
    ## oraculo, B7). La densidad NO se sustituye: el overlay DDSM no anota
    ## densidad por caso. Se deja la densidad predicha, documentado en el panel.
    ##
    pred = carga_rag.prediccion_a_dict_generador(base_pred)
    pred["birads_pred"] = oracle_birads_idx
    pred["birads_confidence"] = 1.0  ## ground truth, no una confianza real del modelo
    pred["malignancy_score"] = oracle_malignancy
    return pred


def _jsonable(x):
    ## Convierte tipos numpy a tipos nativos de Python para poder serializar
    ## a JSON (cache de texto, C4).
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        return float(x)
    return x


def _extract_provenance(attrib):
    ##
    ## Extrae del resultado de atribucion_rag.calcular_atribuciones_rag el
    ## chunk dominante segun Shapley, su documento/pagina y su grounding NLI.
    ## Factorizado para poder llamarse tanto para el reporte predicho como
    ## para la variante oraculo (C5): cada uno puede tener un chunk dominante
    ## distinto porque la query/retrieval interna de generate() depende del
    ## BI-RADS de entrada.
    ##
    shapley_values = attrib["shapley"]["shapley_values"]
    top_chunk_idx = max(shapley_values, key=shapley_values.get)
    top_chunk = attrib["chunks"][top_chunk_idx]
    nli_scores = attrib["nli"]["scores_por_chunk"]
    nli_val = nli_scores.get(top_chunk_idx)
    return {
        "source_document": top_chunk.get("source", "desconocido"),
        "source_page": _jsonable(top_chunk.get("page", "desconocida")),
        "dominant_chunk": (
            top_chunk.get("text", "")[:300]
            + ("..." if len(top_chunk.get("text", "")) > 300 else "")
        ),
        "shapley_value": float(_jsonable(shapley_values[top_chunk_idx])),
        "nli_grounding_score": float(_jsonable(nli_val)) if nli_val is not None else None,
    }


## Marcadores de negacion (J1, ronda de cierre). El alcance de una negacion
## llega hasta el final de la oracion donde aparece el marcador: un termino
## morfologico que cae dentro de ese alcance NO se cuenta como declarado por
## el reporte. Se detecto en la practica: el reporte del caso 1 dice "No se
## identifican masas focales ni calcificaciones agrupadas", y sin este filtro
## "agrupada" quedaba yuxtapuesto como si el reporte la afirmara.
NEGATION_MARKERS = [
    "no se identifican",
    "no se observan",
    "ausencia de",
    "sin evidencia de",
    "descarta",
    "ni",
]
_NEGATION_PATTERNS = [
    re.compile(r"\b" + re.escape(marker) + r"\b", re.IGNORECASE) for marker in NEGATION_MARKERS
]

## Adversativos (L4, ronda de correcciones). El alcance de una negacion NO
## deberia llegar hasta el final de la oracion si aparece un adversativo
## despues del marcador: "No se identifican masas, pero se observan
## calcificaciones pleomorficas" tiene que dejar "pleomorfica" afirmado, no
## negado. El adversativo cierra el alcance en su propia posicion (lo que
## viene despues del adversativo queda fuera de la negacion).
ADVERSATIVE_MARKERS = [
    "pero",
    "sin embargo",
    "aunque",
    "no obstante",
]
_ADVERSATIVE_PATTERNS = [
    re.compile(r"\b" + re.escape(marker) + r"\b", re.IGNORECASE) for marker in ADVERSATIVE_MARKERS
]


def _negated_spans(text):
    ##
    ## Divide el texto en oraciones (separadas por . ! ?) y, para cada una,
    ## busca el marcador de negacion mas temprano. Si hay uno, el tramo
    ## negado va desde ese marcador hasta el final de la oracion, PERO se
    ## cierra antes si aparece un adversativo ("pero", "sin embargo",
    ## "aunque", "no obstante") despues del marcador: lo que sigue al
    ## adversativo ya no esta bajo el alcance de la negacion (L4). Devuelve
    ## una lista de (inicio, fin) en indices del texto completo.
    ##
    spans = []
    for sent_match in re.finditer(r"[^.!?]*[.!?]|[^.!?]+$", text):
        sent = sent_match.group(0)
        sent_start = sent_match.start()
        marker_positions = []
        for pattern in _NEGATION_PATTERNS:
            for m in pattern.finditer(sent):
                marker_positions.append(m.start())
        if marker_positions:
            first_marker = min(marker_positions)
            span_end = len(sent)
            for pattern in _ADVERSATIVE_PATTERNS:
                for m in pattern.finditer(sent):
                    if first_marker < m.start() < span_end:
                        span_end = m.start()
            spans.append((sent_start + first_marker, sent_start + span_end))
    return spans


def _is_negated(pos, spans):
    return any(start <= pos < end for start, end in spans)


def _find_declared_terms(text, terms_es):
    ##
    ## Busca literalmente, en el texto de un reporte, cualquiera de los
    ## terminos en espanol de terms_es (subconjunto de VOCAB_LESION_ES).
    ## Devuelve las coincidencias EXACTAS como aparecen en el texto (preserva
    ## mayusculas), sin duplicados, en el orden en que aparecen, EXCLUYENDO
    ## las que caen dentro del alcance de una negacion (J1: "no se
    ## identifican", "no se observan", "ausencia de", "sin evidencia de",
    ## "descarta", "ni", ver _negated_spans). Se usa SOLO para extraer y
    ## mostrar, nunca para decidir un veredicto (ver E1/E5): un LLM puede
    ## describir la misma morfologia con palabras fuera de esta lista, y
    ## basar un juicio de correctitud en esa busqueda seria enganoso.
    ##
    if not terms_es or not text:
        return []
    unique_terms = sorted(set(terms_es), key=len, reverse=True)
    pattern = re.compile("(" + "|".join(re.escape(t) for t in unique_terms) + ")", re.IGNORECASE)
    negated = _negated_spans(text)
    seen = []
    seen_lower = set()
    for m in pattern.finditer(text):
        if _is_negated(m.start(), negated):
            continue
        match = m.group(0)
        if match.lower() not in seen_lower:
            seen.append(match)
            seen_lower.add(match.lower())
    return seen


## Etiqueta de ausencia (E2). Constatacion, no veredicto: se usa cuando un
## reporte no menciona nada relevante para el campo (tipo de lesion o
## morfologia), y tambien para numero de hallazgos y ubicacion, que el
## generador nunca produce en absoluto.
NO_MENTION_LABEL = "el reporte no menciona este campo"


def _declared_text_or_absence(text, terms_es):
    ##
    ## Texto literal extraido de un reporte para un campo (tipo de lesion o
    ## morfologia), o NO_MENTION_LABEL si no se encontro nada. Ya NO calcula
    ## ningun veredicto (E1): solo extrae para yuxtaponer, sin juzgar.
    ##
    if text is None:
        return None
    terms = _find_declared_terms(text, terms_es)
    return ", ".join(terms) if terms else NO_MENTION_LABEL


def _acr_letter_from_density_label(density_label):
    ## Extrae la letra ACR (A-D) del texto de densidad predicho, ej.
    ## "tejido mamario heterogeneamente denso (ACR C)" -> "C".
    m = re.search(r"\(ACR\s+([A-D])\)", density_label, re.IGNORECASE)
    return m.group(1).upper() if m else None


def _build_concordance_panel(birads_level, pred_pathology, density_label, gt, domain,
                              report_text, oracle_report_text=None):
    ##
    ## Panel de concordancia: dos bloques con logica DISTINTA a proposito.
    ##
    ## Bloque "clasificador" (E4, sin tocar): patologia, BI-RADS y densidad son
    ## comparaciones categoricas exactas (entero contra entero, letra contra
    ## letra) sobre los escalares que produce directamente C8 (exp08). Aqui
    ## "coincide"/"no coincide" es un veredicto valido porque no hay texto
    ## libre de por medio.
    ##
    ## Bloque "hallazgo" (rediseñado, E1-E3): tipo de lesion y morfologia YA
    ## NO llevan veredicto de coincidencia. La version anterior (C1-C2) media
    ## "coincide" por subcadena contra VOCAB_LESION_ES, y eso fallo en la
    ## practica (el reporte oraculo del caso 4 dice "calcificaciones coarsas",
    ## no "groseras", y por subcadena eso se leia como "no coincide" siendo
    ## coherente). En su lugar, se yuxtaponen sin juicio: "lo que dice el
    ## reporte" y "lo que dice la anotacion", una al lado de la otra, para que
    ## quien mira el video sea el que compare. Para el caso 4 (variante
    ## oraculo) ya NO se combinan los dos reportes en un solo veredicto (esa
    ## regla de C1 queda eliminada, E3): cada reporte se yuxtapone por
    ## separado contra la misma anotacion, en columnas rotuladas aparte.
    ## numero de hallazgos y ubicacion son SIEMPRE NO_MENTION_LABEL: el
    ## generador nunca produce esos dos campos, con o sin oraculo.
    ##
    def _cmp(pred, gt_val):
        if pred is None or gt_val is None:
            return "sin ground truth disponible"
        return "coincide" if pred == gt_val else "no coincide"

    gt_density_letter = gt.get("gt_density_letter")
    pred_density_letter = _acr_letter_from_density_label(density_label)
    if gt_density_letter is None:
        density_veredicto = (
            "sin ground truth de densidad en DDSM" if domain == "DDSM"
            else "sin ground truth de densidad disponible"
        )
    else:
        density_veredicto = "coincide" if pred_density_letter == gt_density_letter else "no coincide"

    bloque_clasificador = {
        "patologia": {"veredicto": _cmp(pred_pathology, gt.get("gt_pathology"))},
        "BI-RADS": {"veredicto": _cmp(birads_level, gt.get("gt_birads"))},
        "densidad": {"veredicto": density_veredicto},
    }

    ## Tipo de lesion: DDSM trae lesion_type real por anomalia; VinDr trae
    ## finding_categories (Mass/Calcification/...) de finding_annotations.csv.
    if domain == "DDSM":
        primary = gt["lesions"][0] if gt["lesions"] else None
        gt_lesion_en = [primary["lesion_type"]] if primary else []
    else:
        gt_lesion_en = gt.get("gt_lesion_types_en") or []
    gt_lesion_es = (
        ", ".join(VOCAB_LESION_ES.get(c, c.lower()) for c in gt_lesion_en) if gt_lesion_en else "ninguna"
    )

    ## Morfologia: SOLO DDSM anota SHAPE/MARGINS o TYPE/DISTRIBUTION por
    ## anomalia. VinDr no tiene esas columnas en finding_annotations.csv.
    if domain == "DDSM":
        primary = gt["lesions"][0] if gt["lesions"] else None
        gt_morf_en = [v for v in (primary["morfologia_en"].values() if primary else []) if v]
    else:
        gt_morf_en = []
    gt_morf_es = (
        ", ".join(VOCAB_LESION_ES.get(v, v.lower()) for v in gt_morf_en) if gt_morf_en else "ninguna"
    )

    tipo_pred_text = _declared_text_or_absence(report_text, LESION_TYPE_TERMS_ES)
    tipo_oracle_text = (
        _declared_text_or_absence(oracle_report_text, LESION_TYPE_TERMS_ES)
        if oracle_report_text is not None else None
    )
    morf_pred_text = _declared_text_or_absence(report_text, MORPHOLOGY_TERMS_ES)
    morf_oracle_text = (
        _declared_text_or_absence(oracle_report_text, MORPHOLOGY_TERMS_ES)
        if oracle_report_text is not None else None
    )

    bloque_hallazgo = {
        "numero de hallazgos": {"tipo": "ausencia", "texto": NO_MENTION_LABEL},
        "tipo de lesion": {
            "tipo": "yuxtaposicion",
            "reporte_predicho": tipo_pred_text,
            "reporte_oraculo": tipo_oracle_text,
            "anotacion": gt_lesion_es,
        },
        "morfologia": {
            "tipo": "yuxtaposicion",
            "reporte_predicho": morf_pred_text,
            "reporte_oraculo": morf_oracle_text,
            "anotacion": gt_morf_es,
        },
        "ubicacion": {"tipo": "ausencia", "texto": NO_MENTION_LABEL},
    }

    return {"clasificador": bloque_clasificador, "hallazgo": bloque_hallazgo}


def analyze_case(source, text_cache=None):
    ##
    ## source: uno de los dicts de CASE_SOURCES. kind="vindr" trae "path" (un
    ## DICOM); kind="ddsm" trae "case_id" y "view" (resueltos por
    ## resolve_ddsm_case, que reconcilia los arboles paralelos de imagen y
    ## overlay por ID de caso).
    ##
    ## text_cache: dict cargado por _load_text_cache(), o None. Solo el texto
    ## de los reportes (query, chunks, report_text, provenance, oraculo) se
    ## toma del cache si esta disponible y FORCE_REGENERATE_TEXT es False
    ## (C4): esa es la parte cara y NO determinista del pipeline. El resto
    ## (escalares del clasificador, saliencia, ground truth) se recalcula en
    ## vivo siempre, porque es barato y determinista.
    ##
    kind = source["kind"]
    numero = source["numero"]
    overlay_path = None
    if kind == "vindr":
        image_path = source["path"]
    else:
        image_path, overlay_path = resolve_ddsm_case(source["case_id"], source["view"])

    ## Encoder + dos cabezas (Mammo-CLIP EfficientNet-B5, exp08 / C8). Siempre
    ## en vivo: rapido (una pasada forward) y determinista en modo eval.
    img_tensor = carga_modelo.cargar_imagen(str(image_path), _transform, _device)
    with torch.no_grad():
        base_pred = carga_modelo.obtener_prediccion_base(classifier, img_tensor)

    birads_idx = base_pred["birads_idx"]
    birads_probs = base_pred["birads_probs"][0]

    birads_level = birads_idx + 1
    birads_conf = float(birads_probs[birads_idx].item())
    density_label = DENSITY_DESCRIPTIONS_ES[base_pred["density_idx"]]
    ## malignancy_score = P(BI-RADS 4) + P(BI-RADS 5) (config_xai.MALIGNANT_INDICES)
    malignancy_score = float((birads_probs[3] + birads_probs[4]).item())
    pred_pathology = "MALIGNANT" if malignancy_score >= PATHOLOGY_THRESHOLD else "BENIGN"

    prediction_dict = carga_rag.prediccion_a_dict_generador(base_pred)

    cached_entry = None
    if text_cache is not None and not FORCE_REGENERATE_TEXT:
        cached_entry = text_cache.get(str(numero))

    if cached_entry is not None:
        query = cached_entry["query"]
        retrieved_chunks = cached_entry["retrieved_chunks"]
        report_text = cached_entry["report_text"]
        provenance = cached_entry["provenance"]
        cached_oracle = cached_entry.get("oracle")
    else:
        ## Query morphology-blind + retrieval (PubMedBERT + FAISS).
        query = retriever.build_query_from_findings(
            birads_pred=birads_level, findings=[], density=density_label,
        )
        retrieved_chunks_raw = retriever.retrieve(query, top_k=generator.rag_top_k)

        ## Generacion del reporte real con Qwen2.5-7B-Instruct.
        gen_result = generator.generate(prediction_dict)
        report_text = gen_result["report"]

        ## Provenance/Shapley (XAI/xai/atribucion_rag.calcular_atribuciones_rag).
        attrib = atribucion_rag.calcular_atribuciones_rag(
            generator=generator,
            retriever=retriever,
            llm=_llm,
            tokenizer=_tokenizer,
            nli_model=_nli_model,
            nli_tokenizer=_nli_tokenizer,
            entailment_idx=_entailment_idx,
            prediction_dict=prediction_dict,
        )
        provenance = _extract_provenance(attrib)
        retrieved_chunks = [
            f"[{c['source']}, p.{c['page']}] (score={c['score']:.3f}) {c['text'][:160]}..."
            for c in retrieved_chunks_raw
        ]
        cached_oracle = None

    ## Saliencia: Grad-CAM + Integrated Gradients sobre la cabeza BI-RADS
    ## (XAI/xai/atribucion_clasificador.py). Siempre en vivo.
    saliency = atribucion_clasificador.calcular_atribuciones_imagen(
        model=classifier, img_tensor=img_tensor, head="birads", device=_device,
    )

    ## Ground truth de la columna derecha. Siempre en vivo (parseo de overlay
    ## o de finding_annotations.csv, barato).
    if kind == "vindr":
        gt = _lookup_vindr_ground_truth(image_path)
        domain = "VinDr"
        overlay_data = None
    else:
        gt = _lookup_ddsm_ground_truth(overlay_path)
        domain = "DDSM"
        overlay_data = gt["overlay_data"]

    ## Variante oraculo (B7, solo caso 4). El texto tambien se toma del cache
    ## si esta disponible (C4).
    oracle = None
    if source.get("oraculo"):
        gt_assessment = gt["gt_birads"]
        if cached_oracle is not None:
            oracle = dict(cached_oracle)
        else:
            oracle_birads_idx = gt_assessment - 1
            oracle_malignancy = 1.0 if gt_assessment >= 4 else 0.0
            oracle_prediction_dict = _build_oracle_prediction_dict(
                base_pred, oracle_birads_idx, oracle_malignancy,
            )
            oracle_gen_result = generator.generate(oracle_prediction_dict)

            ## C5: Shapley/provenance de la variante ORACULO, calculado aparte
            ## del reporte predicho: generate() reconstruye su propia query y
            ## retrieval con el BI-RADS de entrada, que aqui es el del ground
            ## truth (4) en vez del predicho (2 para este caso), asi que el
            ## chunk dominante puede ser distinto.
            oracle_attrib = atribucion_rag.calcular_atribuciones_rag(
                generator=generator,
                retriever=retriever,
                llm=_llm,
                tokenizer=_tokenizer,
                nli_model=_nli_model,
                nli_tokenizer=_nli_tokenizer,
                entailment_idx=_entailment_idx,
                prediction_dict=oracle_prediction_dict,
            )
            oracle_provenance = _extract_provenance(oracle_attrib)

            oracle = {
                "birads_level": gt_assessment,
                "report_text": oracle_gen_result["report"],
                "density_note": (
                    "densidad = predicha por el modelo (DDSM no anota densidad por caso)"
                ),
                "provenance": oracle_provenance,
            }

    concordance = _build_concordance_panel(
        birads_level, pred_pathology, density_label, gt, domain,
        report_text, oracle["report_text"] if oracle else None,
    )

    result = {
        "case_number": numero,
        "categoria": source["categoria"],
        "domain": domain,
        "procedencia_html": source["procedencia"],
        "image_path": image_path,
        "birads_level": birads_level,
        "birads_confidence": birads_conf,
        "density_label": density_label,
        "malignancy_score": malignancy_score,
        "pred_pathology": pred_pathology,
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "report_text": report_text,
        "provenance": provenance,
        "saliency": saliency,
        "gt": gt,
        "overlay_data": overlay_data,
        "concordance": concordance,
        "verdict": VERDICT_TEXTS[numero],
    }
    if oracle is not None:
        result["oracle"] = oracle

    ## Entrada de cache (C4): solo el texto, nunca imagen/saliencia/contorno.
    result["_text_cache_entry"] = {
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "report_text": report_text,
        "provenance": provenance,
        "oracle": (
            {
                "birads_level": oracle["birads_level"],
                "report_text": oracle["report_text"],
                "density_note": oracle["density_note"],
                "provenance": oracle["provenance"],
            } if oracle is not None else None
        ),
    }

    return result


<h2>Funcion de renderizado: comparador de dos columnas</h2>
<p>Toma el diccionario de <code>analyze_case</code> y arma el panel completo, en este
orden de lectura vertical: encabezado de caso (numero, categoria, origen,
procedencia y los tres escalares del modelo), las dos imagenes (izquierda: salida
del modelo con saliencia; derecha: ground truth), el panel de concordancia, la linea
de veredicto, el reporte generado con la morfologia resaltada, y al final la
evidencia de auditoria. Mantiene el mecanismo de <code>target.value = html</code>
(sin <code>Output()</code>/<code>clear_output()</code>).</p>

In [6]:
## Colores consistentes con la Figura 1 del paper: teal para el modulo de vision,
## naranja para el puente de escalares, morado para el modulo de lenguaje/auditoria.
TEAL = "#2E5F6E"
ORANGE = "#C05A0A"
PURPLE = "#4A3B5C"
GT_BOX_COLOR = "#39FF14"  ## verde de la Figura 1: "ground-truth lesion annotation, reference only"
## V4 (ronda de correcciones): GT_BOX_COLOR es verde neon, funciona como
## borde de caja sobre una imagen oscura pero es ilegible como texto sobre
## fondo blanco (contraste medido: 1.36:1 contra #FFFFFF, muy por debajo del
## minimo WCAG AA de 4.5:1 para texto normal). GT_TEXT_COLOR es una variante
## mas oscura del mismo tono (contraste medido: 5.13:1), para usar en
## etiquetas/bordes/fondos de texto; GT_BOX_COLOR se reserva para el borde de
## las cajas sobre imagen, donde el verde neon si funciona.
GT_TEXT_COLOR = "#2E7D32"
HIGHLIGHT_BG = "#FFE58A"  ## fondo de resaltado de fragmentos morfologicos (B14)

import io
import re
import base64
from PIL import Image


def _fig_to_b64(fig, dpi=110):
    ## Serializa una figura de matplotlib a PNG en memoria y la codifica en base64.
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi)
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode("ascii")


def _pil_to_b64(pil_img):
    ## Codifica una imagen PIL en base64 sin pasar por matplotlib (para el
    ## recorte ampliado de la lesion, B3).
    buf = io.BytesIO()
    pil_img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii")


def _left_image_html(result):
    ## Columna izquierda: SALIDA DEL MODELO. Imagen con mapa de saliencia
    ## Grad-CAM sobre la cabeza BI-RADS, superpuesta como heatmap. Integrated
    ## Gradients se calculo en analyze_case() (B2 exige ambos), pero se reporta
    ## como escalar (atribucion total) en lugar de superponerlo: dos heatmaps
    ## a la vez saturan la imagen y dificultan la lectura sin narracion.
    pil_img = carga_modelo.load_image_as_pil(str(result["image_path"]))
    img = np.array(pil_img.convert("L"))

    gradcam = result["saliency"]["gradcam"]
    cam_norm = gradcam / (gradcam.max() + 1e-8)
    cam_img = Image.fromarray((cam_norm * 255).astype(np.uint8))
    cam_img = cam_img.resize((img.shape[1], img.shape[0]), Image.BILINEAR)
    cam_resized = np.array(cam_img).astype(np.float32) / 255.0

    fig, ax = plt.subplots(figsize=(5, 6.5))
    ax.imshow(img, cmap="gray")
    ax.imshow(cam_resized, cmap="jet", alpha=0.42)
    ax.axis("off")
    ax.set_title("Salida del modelo (Grad-CAM, cabeza BI-RADS)", fontsize=13)
    plt.tight_layout()
    b64 = _fig_to_b64(fig)

    ig_total = float(np.abs(result["saliency"]["ig"]).sum())
    caption = (
        f"Integrated Gradients tambien calculado (atribucion total = {ig_total:.1f}); "
        "no se superpone junto al Grad-CAM para no saturar el heatmap."
    )
    return f'''
    <div>
      <img src="data:image/png;base64,{b64}" style="width:660px; max-width:100%;">
      <div style="font-size:13px; color:#666; margin-top:4px;">{caption}</div>
    </div>
    '''


def _right_image_html(result):
    ## Columna derecha: GROUND TRUTH. VinDr: caja anotada. DDSM: contorno
    ## reconstruido por LesionContour (ddsm_overlay.draw_lesion_contour, NO el
    ## bbox), mas un recorte ampliado (crop_lesion_roi) y la tabla de
    ## descriptores morfologicos (B3).
    pil_img = carga_modelo.load_image_as_pil(str(result["image_path"]))
    gt = result["gt"]
    table_html = ""

    if result["domain"] == "VinDr":
        img = np.array(pil_img.convert("L"))
        fig, ax = plt.subplots(figsize=(5, 6.5))
        ax.imshow(img, cmap="gray")
        boxes = gt.get("gt_boxes") or []
        box_labels_en = gt.get("gt_box_labels_en") or []
        if boxes:
            ref_h, ref_w = gt["gt_box_ref_size"]
            scale_x = img.shape[1] / ref_w
            scale_y = img.shape[0] / ref_h
            ## Paleta de colores por caja (O2): con una sola caja se mantiene
            ## GT_BOX_COLOR (verde, igual que siempre); con mas de una, cada
            ## caja y su rotulo comparten un color distinto para poder
            ## distinguirlas -hoy dos cajas identicas en verde eran
            ## indistinguibles entre si en el panel.
            box_color_palette = [GT_BOX_COLOR, "#FF6B35", "#00BFFF", "#FF1493"]
            for box_idx, (xmin, ymin, xmax, ymax) in enumerate(boxes):
                box_color = box_color_palette[box_idx % len(box_color_palette)]
                rect = mpatches.Rectangle(
                    (xmin * scale_x, ymin * scale_y),
                    (xmax - xmin) * scale_x, (ymax - ymin) * scale_y,
                    linewidth=2, edgecolor=box_color, facecolor="none",
                )
                ax.add_patch(rect)
                ## Rotulo de la caja (O2): finding_category de ESTA fila,
                ## traducida con VOCAB_LESION_ES. Si no hay categoria
                ## asociada a esta caja especifica, no se dibuja ningun
                ## rotulo -se prefiere sin rotulo a uno adivinado.
                cats_en = box_labels_en[box_idx] if box_idx < len(box_labels_en) else []
                if cats_en:
                    label_es = ", ".join(
                        VOCAB_LESION_ES.get(c, c.lower().replace("_", " ")) for c in cats_en
                    )
                    ax.text(
                        xmin * scale_x, max(0, ymin * scale_y - 8),
                        label_es, color=box_color, fontsize=14, fontweight="bold",
                        ha="left", va="bottom",
                    )
        ax.axis("off")
        ax.set_title("Ground truth (VinDr, caja anotada)", fontsize=13)
        plt.tight_layout()
        b64 = _fig_to_b64(fig)
        note = (
            f"Hallazgos anotados: {gt['n_findings']}" if boxes
            else "Sin hallazgo anotado (BI-RADS 1, No Finding)"
        )
    else:
        overlay_data = result["overlay_data"]
        contour_img = ddsm_overlay.draw_lesion_contour(pil_img, overlay_data)
        fig, ax = plt.subplots(figsize=(5, 6.5))
        ax.imshow(np.array(contour_img))
        ax.axis("off")
        ax.set_title("Ground truth (DDSM, contorno del overlay)", fontsize=13)
        plt.tight_layout()
        b64 = _fig_to_b64(fig)
        note = f"Anomalias en el overlay (TOTAL_ABNORMALITIES): {gt['n_findings']}"

        crop_html = ""
        first_lesion = gt["lesions"][0] if gt["lesions"] else None
        if first_lesion and first_lesion["contours"]:
            roi, _box = ddsm_overlay.crop_lesion_roi(pil_img, first_lesion["contours"][0], padding=30)
            roi_b64 = _pil_to_b64(roi)
            crop_html = f'''
            <div style="margin-top:10px;">
              <img src="data:image/png;base64,{roi_b64}" style="width:280px;">
              <div style="font-size:13px; color:#666;">Recorte ampliado de la lesion</div>
            </div>
            '''

        rows = []
        for les in gt["lesions"]:
            morf_es = ", ".join(
                f"{k}={v} ({VOCAB_LESION_ES.get(v, v)})"
                for k, v in les["morfologia_en"].items() if v
            )
            lesion_type_es = VOCAB_LESION_ES.get(les["lesion_type"], les["lesion_type"])
            rows.append(
                f"<tr><td>{lesion_type_es} ({les['lesion_type']})</td>"
                f"<td>{morf_es}</td><td>{les['assessment']}</td>"
                f"<td>{les['subtlety']}</td><td>{les['pathology']}</td></tr>"
            )
        table_html = f'''
        <table style="font-size:18px; border-collapse:collapse; margin-top:10px;">
          <tr style="font-weight:bold;">
            <td>Tipo de lesion</td><td>Morfologia</td><td>ASSESSMENT</td>
            <td>SUBTLETY</td><td>PATHOLOGY</td>
          </tr>
          {"".join(rows)}
        </table>
        <div style="font-size:18px; margin-top:6px;">
          TOTAL_ABNORMALITIES: {gt["n_findings"]}
        </div>
        {crop_html}
        '''

    return f'''
    <div>
      <img src="data:image/png;base64,{b64}" style="width:660px; max-width:100%;">
      <div style="font-size:13px; color:#666; margin-top:4px;">{note}</div>
      {table_html}
    </div>
    '''


def _header_html(result):
    ## Encabezado de caso (B11): numero, categoria, origen, procedencia (B9),
    ## y los tres escalares que produce el modelo. Tipografia grande.
    return f'''
    <div style="margin-bottom:16px;">
      <div style="font-size:28px; font-weight:bold; color:{TEAL};">
        CASO {result["case_number"]} &mdash; {result["categoria"].upper()} &mdash; {result["domain"]}
      </div>
      <div style="font-size:18px; color:#444; margin-top:6px;">
        {result["procedencia_html"]}
      </div>
      <div style="display:flex; gap:32px; margin-top:14px;">
        <div>
          <div style="font-size:12px; letter-spacing:1px; color:#666;">BI-RADS PREDICHO</div>
          <div style="font-size:36px; font-weight:bold; color:{TEAL};">{result["birads_level"]}</div>
          <div style="font-size:12px; color:#666;">confianza {result["birads_confidence"]:.3f}</div>
        </div>
        <div>
          <div style="font-size:12px; letter-spacing:1px; color:#666;">DENSIDAD ACR PREDICHA</div>
          <div style="font-size:36px; font-weight:bold; color:{TEAL};">{result["density_label"]}</div>
        </div>
        <div>
          <div style="font-size:12px; letter-spacing:1px; color:#666;">MALIGNANCY SCORE</div>
          <div style="font-size:36px; font-weight:bold; color:{ORANGE};">{result["malignancy_score"]:.3f}</div>
        </div>
      </div>
    </div>
    '''


def _concordance_field_html(campo, datos):
    ## Un campo de "ausencia" (numero de hallazgos, ubicacion): una linea,
    ## texto fijo (E2). Un campo de "yuxtaposicion" (tipo de lesion,
    ## morfologia): dos lineas grandes, sin veredicto (E1) - "lo que dice el
    ## reporte" y "lo que dice la anotacion", una debajo de la otra para
    ## comparacion inmediata. Si el caso tiene variante oraculo (reporte_oraculo
    ## no es None), se muestran DOS columnas por separado -predicho y
    ## oraculo-, cada una yuxtapuesta contra la misma anotacion; nunca se
    ## combinan en un solo veredicto (E3, regla de C1 eliminada).
    if "tipo" not in datos:
        ## Bloque 1 (clasificador): solo veredicto categorico exacto (E4,
        ## sin tocar). No tiene clave "tipo": una sola linea campo/veredicto.
        return f'''
        <div style="margin-top:10px; display:flex; justify-content:space-between; max-width:680px;">
          <div>{campo}</div>
          <div style="font-weight:bold;">{datos["veredicto"]}</div>
        </div>
        '''

    if datos.get("tipo") == "ausencia":
        return f'''
        <div style="margin-top:10px; font-size:20px;">
          <span style="font-weight:bold;">{campo}:</span> {datos["texto"]}
        </div>
        '''

    def _par(reporte_texto, anotacion_texto):
        return f'''
        <div style="font-size:20px; margin-top:4px;">lo que dice el reporte: {reporte_texto}</div>
        <div style="font-size:20px;">lo que dice la anotacion: {anotacion_texto}</div>
        '''

    if datos.get("reporte_oraculo") is not None:
        columna_pred = f'''
        <div style="flex:1;">
          <div style="font-weight:bold; font-size:15px; color:{PURPLE};">reporte con escalares predichos</div>
          {_par(datos["reporte_predicho"], datos["anotacion"])}
        </div>
        '''
        columna_oraculo = f'''
        <div style="flex:1;">
          <div style="font-weight:bold; font-size:15px; color:{PURPLE};">reporte con escalares del ground truth</div>
          {_par(datos["reporte_oraculo"], datos["anotacion"])}
        </div>
        '''
        cuerpo = f'<div style="display:flex; gap:32px; margin-top:6px;">{columna_pred}{columna_oraculo}</div>'
    else:
        cuerpo = _par(datos["reporte_predicho"], datos["anotacion"])

    return f'''
    <div style="margin-top:16px;">
      <div style="font-weight:bold; font-size:20px;">{campo}</div>
      {cuerpo}
    </div>
    '''


def _concordance_html(concordance):
    ## Panel de concordancia, fuente minima 20px (B16), dividido en dos
    ## bloques con encabezado propio (C3): "Salida del clasificador" (los tres
    ## escalares que produce C8 directamente, con veredicto categorico exacto,
    ## E4) y "Descripcion del hallazgo" (yuxtaposicion sin veredicto entre lo
    ## que dice el reporte y lo que dice la anotacion, E1-E3). Separar los dos
    ## bloques evita que la falta de juicio del bloque 2 se lea como falta de
    ## rigor del bloque 1.
    bloque1_html = "".join(
        _concordance_field_html(campo, datos) for campo, datos in concordance["clasificador"].items()
    )
    bloque2_html = "".join(
        _concordance_field_html(campo, datos) for campo, datos in concordance["hallazgo"].items()
    )
    return f'''
    <div style="margin-top:24px; font-size:20px;">
      <div style="font-weight:bold; color:{PURPLE};">PANEL DE CONCORDANCIA</div>
      <div style="margin-top:14px; font-weight:bold; color:{TEAL};">BLOQUE 1: SALIDA DEL CLASIFICADOR</div>
      {bloque1_html}
      <div style="margin-top:20px; font-weight:bold; color:{TEAL};">BLOQUE 2: DESCRIPCION DEL HALLAZGO</div>
      {bloque2_html}
    </div>
    '''


def _verdict_html(verdict_text):
    ## Linea de veredicto (B12): una sola frase en cuerpo grande.
    return f'''
    <div style="margin-top:18px; font-size:24px; font-weight:bold; color:{TEAL};">
      {verdict_text}
    </div>
    '''


def _highlight_morphology(text):
    ## Resalta con fondo de color cualquier termino en espanol de
    ## VOCAB_LESION_ES que aparezca en el reporte generado (B14). Sirve para
    ## que sea visible de un vistazo que ese texto viene de la literatura (RAG),
    ## no de una prediccion morfologica del modelo (ver panel de concordancia).
    terms = sorted(set(VOCAB_LESION_ES.values()), key=len, reverse=True)
    pattern = re.compile("(" + "|".join(re.escape(t) for t in terms) + ")", re.IGNORECASE)
    return pattern.sub(lambda m: f'<mark style="background:{HIGHLIGHT_BG};">{m.group(1)}</mark>', text)


def _report_block_html(label, text):
    highlighted = _highlight_morphology(text)
    return f'''
    <div>
      <div style="font-size:18px; font-weight:bold; color:{PURPLE};">{label}</div>
      <div style="white-space:pre-wrap; font-size:15px; margin-top:6px;">{highlighted}</div>
    </div>
    '''


def _contradiction_callout(result):
    ##
    ## E7: si el reporte con escalares predichos afirma ausencia de hallazgos
    ## sospechosos pese a que el propio clasificador asigno patologia
    ## MALIGNANT (malignancy_score por encima del umbral), lo declara
    ## explicitamente en vez de dejar que la contradiccion pase inadvertida.
    ## La condicion se evalua sobre result["report_text"] tal como llego (del
    ## cache si habia cache, C4): nunca dispara una regeneracion, asi que no
    ## depende del sampling no determinista del LLM.
    ##
    if result["pred_pathology"] != "MALIGNANT":
        return ""
    if "no se identifican hallazgos sospechosos" not in result["report_text"].lower():
        return ""
    return f'''
    <div style="margin-top:14px; padding:12px; border:2px solid {ORANGE}; background:#FFF4EA; font-size:15px;">
      El texto generado afirma que no se identifican hallazgos sospechosos. El
      propio clasificador asigno a este caso un malignancy_score de {result["malignancy_score"]:.3f},
      por encima del umbral, y la patologia confirmada por biopsia es maligna.
      El reporte contradice al clasificador que lo alimenta.
    </div>
    '''


def _oracle_chunk_connector_html(result):
    ##
    ## E6: el texto morfologico del reporte oraculo y su chunk dominante por
    ## Shapley tienen que quedar en el mismo fotograma -columna adyacente al
    ## reporte oraculo, no separado abajo en el panel de auditoria. Esta
    ## funcion arma esa columna; se llama solo para el caso con variante
    ## oraculo (numero 4). El panel de auditoria al final conserva el detalle
    ## completo (los 3 chunks, la query); esto es la conexion visual
    ## inmediata, no un reemplazo de esa auditoria.
    ##
    oprov = result["oracle"].get("provenance")
    if not oprov:
        return ""
    return f'''
    <div style="flex:1; border:2px solid {ORANGE}; background:#FFF4EA; padding:12px; font-size:14px;">
      <div style="font-weight:bold; color:{PURPLE};">CHUNK DOMINANTE DE ESTE REPORTE (Shapley)</div>
      <div style="margin-top:8px;">
        documento: {oprov["source_document"]}<br>
        pagina: {oprov["source_page"]}<br>
        <i>{oprov["dominant_chunk"]}</i><br>
        valor de Shapley: {oprov["shapley_value"]:.3f}<br>
        grounding NLI del reporte contra este chunk: {oprov["nli_grounding_score"]:.3f}
      </div>
    </div>
    '''


def _report_section_html(result):
    ## Reporte generado con morfologia resaltada. Caso sin variante oraculo:
    ## una sola columna, con el callout de contradiccion (E7) si aplica. Caso
    ## con variante oraculo (numero 4): 3 columnas adyacentes -reporte
    ## predicho (con su callout de contradiccion), reporte oraculo, y el chunk
    ## dominante del oraculo (E6)- para que el texto y su procedencia queden
    ## en el mismo fotograma, no separados por scroll.
    note = (
        "Nota: el generador no recibio la imagen al escribir esto. Solo recibio "
        "los escalares BI-RADS/densidad y los chunks recuperados; los fragmentos "
        "resaltados vienen de la literatura (RAG), no de una prediccion de "
        "morfologia del modelo."
    )
    contradiction_html = _contradiction_callout(result)

    if "oracle" not in result:
        body = _report_block_html("REPORTE GENERADO", result["report_text"])
        return f'''
        <div style="margin-top:20px; padding:16px; border:1px solid #ddd;">
          {body}
          {contradiction_html}
          <div style="font-size:13px; color:#a05a00; margin-top:12px; font-style:italic;">{note}</div>
        </div>
        '''

    left = _report_block_html("ESCALARES PREDICHOS POR EL MODELO", result["report_text"])
    right = _report_block_html("ESCALARES DEL GROUND TRUTH", result["oracle"]["report_text"])
    connector_html = _oracle_chunk_connector_html(result)

    oracle_note = (
        "Los dos reportes se yuxtaponen, no se combinan (E3): cada uno se lee "
        "por separado. La morfologia fabricada por el generador es identica en "
        "ambas columnas -en ninguna de las dos corridas recibio hallazgos "
        "morfologicos reales, findings=[] en ambos casos, ver src/rag.py-; solo "
        "cambia el BI-RADS de entrada. " + result["oracle"]["density_note"] + "."
    )

    return f'''
    <div style="margin-top:20px; padding:16px; border:1px solid #ddd;">
      <div style="display:flex; gap:24px; align-items:flex-start;">
        <div style="flex:1;">{left}{contradiction_html}</div>
        <div style="flex:1;">{right}</div>
        {connector_html}
      </div>
      <div style="font-size:14px; color:#666; margin-top:10px;">{oracle_note}</div>
      <div style="font-size:13px; color:#a05a00; margin-top:12px; font-style:italic;">{note}</div>
    </div>
    '''


def _audit_html(result):
    ## Evidencia de auditoria, al final del panel (B17). "chunk dominante
    ## (Shapley)" reemplaza la etiqueta anterior "fragmento trasladado" (B10).
    ## Escalares formateados a 3 decimales (B10). Para el caso con variante
    ## oraculo, se agrega el bloque de procedencia completo de la variante
    ## oraculo, con el mismo destaque de C5 (borde naranja si el chunk
    ## dominante es la referencia ACR BI-RADS de calcificaciones coarsas
    ## heterogeneas). La conexion visual inmediata ya vive junto al reporte
    ## (E6, _oracle_chunk_connector_html); este bloque es el detalle completo
    ## para quien quiera auditar despues.
    prov = result["provenance"]
    chunks_html = "".join(f"<li>{c}</li>" for c in result["retrieved_chunks"])
    base_html = f'''
    <div style="margin-top:24px; padding:16px; background:#F7F5F9;">
      <div style="font-size:14px; letter-spacing:1px; color:{PURPLE}; font-weight:bold;">
        EVIDENCIA DE AUDITORIA
      </div>
      <div style="margin-top:10px; font-size:14px;">
        <b>Query de recuperacion (morphology-blind):</b><br>{result["query"]}
      </div>
      <div style="margin-top:10px; font-size:14px;">
        <b>Chunks recuperados:</b>
        <ul>{chunks_html}</ul>
      </div>
      <div style="margin-top:10px; font-size:14px;">
        <b>Procedencia del reporte predicho (Shapley / provenance):</b><br>
        documento: {prov["source_document"]}<br>
        pagina: {prov["source_page"]}<br>
        chunk dominante (Shapley): <i>{prov["dominant_chunk"]}</i><br>
        valor de Shapley: {prov["shapley_value"]:.3f}<br>
        grounding NLI: {prov["nli_grounding_score"]:.3f}
      </div>
    </div>
    '''

    oracle_html = ""
    if "oracle" in result and result["oracle"].get("provenance"):
        oprov = result["oracle"]["provenance"]
        dominant_lower = oprov["dominant_chunk"].lower()
        doc_lower = oprov["source_document"].lower()
        is_calc_ref = "birads" in doc_lower and (
            "calcification" in dominant_lower or "coarse" in dominant_lower
        )
        highlight_style = (
            f"border:2px solid {ORANGE}; background:#FFF4EA;" if is_calc_ref
            else "border:1px solid #ddd; background:#F7F5F9;"
        )
        callout_html = (
            f'<div style="margin-top:8px; font-weight:bold; color:{ORANGE};">'
            "Este chunk es la referencia ACR BI-RADS de calcificaciones coarsas "
            "heterogeneas: es la evidencia que sustenta el BI-RADS del ground "
            "truth en la variante oraculo.</div>"
        ) if is_calc_ref else ""
        oracle_html = f'''
        <div style="margin-top:16px; padding:16px; {highlight_style}">
          <div style="font-size:14px; letter-spacing:1px; color:{PURPLE}; font-weight:bold;">
            PROCEDENCIA DEL REPORTE ORACULO (Shapley / provenance)
          </div>
          <div style="margin-top:10px; font-size:14px;">
            documento: {oprov["source_document"]}<br>
            pagina: {oprov["source_page"]}<br>
            chunk dominante (Shapley): <i>{oprov["dominant_chunk"]}</i><br>
            valor de Shapley: {oprov["shapley_value"]:.3f}<br>
            grounding NLI del reporte contra este chunk: {oprov["nli_grounding_score"]:.3f}
          </div>
          {callout_html}
        </div>
        '''

    return base_html + oracle_html


def render_result(result, target=None):
    ## Se conserva el mecanismo de escribir target.value directamente (sin
    ## Output()/clear_output()). Orden de lectura vertical exacto (B17):
    ## encabezado, las dos imagenes, concordancia, veredicto, reporte,
    ## evidencia de auditoria al final.
    header_html = _header_html(result)
    images_html = f'''
    <div style="display:flex; gap:24px; margin-top:16px; flex-wrap:wrap;">
      {_left_image_html(result)}
      {_right_image_html(result)}
    </div>
    '''
    concordance_html = _concordance_html(result["concordance"])
    verdict_html = _verdict_html(result["verdict"])
    report_html = _report_section_html(result)
    audit_html = _audit_html(result)

    combined_html = (
        header_html + images_html + concordance_html
        + verdict_html + report_html + audit_html
    )

    if target is not None:
        target.value = combined_html
    else:
        display(HTML(combined_html))


<h2>Precomputo de los 4 casos fijos</h2>
<p>B15: los 4 casos se calculan una sola vez al ejecutar esta celda (clasificador,
saliencia, generador RAG, Shapley/NLI, y la variante oraculo del caso 4). Los
botones de la celda de interfaz solo pintan resultados ya en cache: cero latencia y
cero riesgo de excepcion durante la toma. El progreso se imprime de forma sobria
(sin barras de progreso ni caracteres especiales).</p>

In [7]:
print("Precomputando los 4 casos fijos.")

_text_cache = _load_text_cache()
if FORCE_REGENERATE_TEXT:
    print("FORCE_REGENERATE_TEXT=True: se ignora el cache de texto y se regenera todo.")
elif _text_cache:
    print(f"Cache de texto encontrado en {CASE_CACHE_PATH}: se reutiliza el texto de los reportes.")
else:
    print(f"No hay cache de texto en {CASE_CACHE_PATH}: se genera todo (LLM de 7B + Shapley).")

CASE_RESULTS = {}
_new_cache_entries = {}
for _numero in sorted(CASE_SOURCES):
    _source = CASE_SOURCES[_numero]
    print(f"Caso {_numero} ({_source['categoria']}, {_source['origen']}): iniciando")
    _result = analyze_case(_source, text_cache=_text_cache)
    _new_cache_entries[str(_numero)] = _result.pop("_text_cache_entry")
    CASE_RESULTS[_numero] = _result
    print(f"Caso {_numero}: completado")

_save_text_cache(_new_cache_entries)
print(f"Cache de texto guardado en {CASE_CACHE_PATH}")
print("Precomputacion completa. Los 4 resultados quedan en cache en CASE_RESULTS.")


Precomputando los 4 casos fijos.
Cache de texto encontrado en outputs/demo_cache/case_results_cache.json: se reutiliza el texto de los reportes.
Caso 1 (maligno, VinDr-Mammo): iniciando


/usr/local/lib/python3.11/dist-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: 'b6bc74e44e775f7ba981fb608fc16ef6'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)


Caso 1: completado
Caso 2 (normal, VinDr-Mammo): iniciando


/usr/local/lib/python3.11/dist-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: 'd20fff9fa12bae900281b3153c48fd71'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)


Caso 2: completado
Caso 3 (benigno, DDSM): iniciando
Caso 3: completado
Caso 4 (maligno, DDSM): iniciando
Caso 4: completado
Cache de texto guardado en outputs/demo_cache/case_results_cache.json
Precomputacion completa. Los 4 resultados quedan en cache en CASE_RESULTS.


<h2>Panel reducido (demo de un minuto, W1-W5)</h2>
<p>Version reducida para grabacion: cuatro bloques y nada mas, en este orden:
encabezado (numero de caso, categoria, origen, procedencia), las dos
imagenes (izquierda limpia sin saliencia; derecha con contorno/cajas de
ground truth en CIAN -codigo de color unico para todo el demo, W1/W5,
reemplaza el rojo/verde por patologia y la paleta multicolor por caja de
rondas anteriores- con trazo doble negro+cian y relleno translucido (W2/W3),
mas un recorte ampliado de la lesion para los casos DDSM (W4)), el BLOQUE 1
DE SALIDA DEL CLASIFICADOR (patologia, BI-RADS y densidad, formato de dos
lineas pegadas, sin veredicto de coincidencia), y el BLOQUE 2 DE DESCRIPCION
DEL HALLAZGO (reporte generado integro frente a la anotacion traducida, en
una linea). Sin frase de veredicto. Sin restriccion de altura de fotograma.
Todo lo que sale del panel vive en
<code>outputs/demo_paneles/notas_defensa.html</code>, documento aparte que no
se renderiza en el notebook.</p>


In [8]:
## Panel reducido para el demo de un minuto (Q1-Q2 corregido por R1-R6, ronda
## de cierre). Cuatro bloques y nada mas, en este orden: encabezado (numero,
## categoria, origen, procedencia -sin cambios, R6), las dos imagenes
## (izquierda limpia, sin saliencia; derecha con contorno o cajas de ground
## truth y sus rotulos, sin tabla de descriptores ni recorte), el BLOQUE 1 de
## SALIDA DEL CLASIFICADOR (patologia, BI-RADS y densidad -esta ultima solo
## si hay ground truth, Q2- en el MISMO formato de dos lineas pegadas que el
## Bloque 2, R2/R4: ya NO es "coincide"/"no coincide"), y el BLOQUE 2 de
## DESCRIPCION DEL HALLAZGO (tipo de lesion y morfologia, formato existente
## sin cambios, R3). La frase de veredicto YA NO se renderiza en el panel
## (R1): VERDICT_TEXTS sigue existiendo tal cual en el notebook, solo dejo de
## usarse aqui. Los dos bloques conservan su encabezado de rotulo (R5): son
## nombres de contenido, no explicacion.
##
## Todo lo que sale del panel (numero de hallazgos, ubicacion, reporte
## completo, callout de contradiccion, variante oraculo, auditoria, cierre,
## y ahora tambien la frase de veredicto de los 4 casos) vive en
## outputs/demo_paneles/notas_defensa.html -las 4 frases de veredicto ya
## estaban ahi desde que notas_defensa.html se genero reusando render_result()
## completo para los 4 casos, no hace falta agregarlas de nuevo.
##
## No reemplaza render_result(): esa funcion sigue existiendo tal cual y es la
## que se usa para armar notas_defensa.html (con el detalle completo de los 4
## casos, incluido el caso 1).


def _header_reduced_html(result):
    ## Bloque a: numero de caso, categoria, origen, procedencia (R6, sin
    ## cambios). Sin el bloque de escalares del clasificador (BI-RADS/
    ## densidad/malignancy score) que si trae _header_html completo.
    return f'''
    <div style="margin-bottom:16px;">
      <div style="font-size:28px; font-weight:bold; color:{TEAL};">
        CASO {result["case_number"]} &mdash; {result["categoria"].upper()} &mdash; {result["domain"]}
      </div>
      <div style="font-size:18px; color:#444; margin-top:6px;">
        {result["procedencia_html"]}
      </div>
    </div>
    '''


def _left_image_clean_html(result):
    ## Bloque b, columna izquierda: imagen limpia, sin mapa de saliencia.
    pil_img = carga_modelo.load_image_as_pil(str(result["image_path"]))
    img = np.array(pil_img.convert("L"))

    fig, ax = plt.subplots(figsize=(5, 6.5))
    ax.imshow(img, cmap="gray")
    ax.axis("off")
    ax.set_title("Imagen", fontsize=13)
    plt.tight_layout()
    b64 = _fig_to_b64(fig)

    return f'''
    <div>
      <img src="data:image/png;base64,{b64}" style="width:660px; max-width:100%;">
    </div>
    '''


## GT_CYAN (W1, ronda de correcciones): color unico de ground truth para
## TODO el demo (imagenes VinDr y DDSM por igual, W5), reemplaza el
## rojo/verde por patologia que traia draw_lesion_contour() y la paleta
## multicolor por caja que traian los boxes VinDr. Motivo: la imagen esta en
## escala de grises, asi que un color puro fuera del eje gris (ni R=G=B)
## maximiza el contraste sobre cualquier tejido; ademas el canal rojo es el
## que peor sobrevive a la compresion de video (submuestreo de crominancia
## tipico en 4:2:0), asi que se evita depender de el -cian es G+B puro, sin
## componente roja. Los rotulos de texto junto a cada caja/contorno (ya
## coloreados por su propio contenido, no por color) siguen distinguiendo
## cada hallazgo cuando hay mas de uno; el color de la caja ya no necesita
## hacerlo.
import matplotlib.patheffects as pe

GT_CYAN = "#00FFFF"


def _draw_gt_shape(ax, xy_points):
    ##
    ## W2: trazo doble -linea negra mas gruesa por debajo, linea cian mas
    ## fina encima- para que el contorno se distinga tanto sobre tejido claro
    ## como sobre el fondo negro de la imagen. W3: relleno translucido cian
    ## (alpha 0.12) del area encerrada, que marca el area sin ocultar la
    ## textura de la lesion. Tres patches superpuestos porque
    ## matplotlib.patches.Polygon no permite alpha distinto para el borde y
    ## el relleno en un solo patch.
    ##
    borde_negro = mpatches.Polygon(
        xy_points, closed=True, edgecolor="black", facecolor="none", linewidth=5,
    )
    relleno_cian = mpatches.Polygon(
        xy_points, closed=True, edgecolor="none", facecolor=GT_CYAN, alpha=0.12,
    )
    borde_cian = mpatches.Polygon(
        xy_points, closed=True, edgecolor=GT_CYAN, facecolor="none", linewidth=2,
    )
    ax.add_patch(borde_negro)
    ax.add_patch(relleno_cian)
    ax.add_patch(borde_cian)


def _right_image_reduced_html(result):
    ## Bloque b, columna derecha: contorno/cajas de ground truth en cian
    ## (W1-W3, W5), con rotulos de texto. Para DDSM se agrega un recorte
    ## ampliado de la lesion (W4, crop_lesion_roi con margen de contexto)
    ## junto a la imagen de contorno completo: el contorno completo situa la
    ## lesion, el recorte la muestra.
    pil_img = carga_modelo.load_image_as_pil(str(result["image_path"]))
    gt = result["gt"]

    if result["domain"] == "VinDr":
        img = np.array(pil_img.convert("L"))
        fig, ax = plt.subplots(figsize=(5, 6.5))
        ax.imshow(img, cmap="gray")
        boxes = gt.get("gt_boxes") or []
        box_labels_en = gt.get("gt_box_labels_en") or []
        if boxes:
            ref_h, ref_w = gt["gt_box_ref_size"]
            scale_x = img.shape[1] / ref_w
            scale_y = img.shape[0] / ref_h
            for box_idx, (xmin, ymin, xmax, ymax) in enumerate(boxes):
                x0, y0 = xmin * scale_x, ymin * scale_y
                x1, y1 = xmax * scale_x, ymax * scale_y
                _draw_gt_shape(ax, [(x0, y0), (x1, y0), (x1, y1), (x0, y1)])
                cats_en = box_labels_en[box_idx] if box_idx < len(box_labels_en) else []
                if cats_en:
                    label_es = ", ".join(
                        VOCAB_LESION_ES.get(c, c.lower().replace("_", " ")) for c in cats_en
                    )
                    ax.text(
                        x0, max(0, y0 - 8),
                        label_es, color=GT_CYAN, fontsize=14, fontweight="bold",
                        ha="left", va="bottom",
                        path_effects=[pe.withStroke(linewidth=3, foreground="black")],
                    )
        ax.axis("off")
        ax.set_title("Ground truth (VinDr, caja anotada)", fontsize=13)
        plt.tight_layout()
        b64 = _fig_to_b64(fig)
        return f'''
        <div>
          <img src="data:image/png;base64,{b64}" style="width:660px; max-width:100%;">
        </div>
        '''

    ## DDSM: contorno reconstruido (LesionContour.points), no el bbox.
    overlay_data = result["overlay_data"]
    img_gray = np.array(pil_img.convert("L"))
    fig, ax = plt.subplots(figsize=(5, 6.5))
    ax.imshow(img_gray, cmap="gray")
    for lesion in overlay_data.lesions:
        for contour in lesion.contours:
            if len(contour.x) < 2:
                continue
            _draw_gt_shape(ax, list(zip(contour.x.tolist(), contour.y.tolist())))
    ax.axis("off")
    ax.set_title("Ground truth (DDSM, contorno del overlay)", fontsize=13)
    plt.tight_layout()
    b64_contorno = _fig_to_b64(fig)

    ## W4: recorte ampliado de la primera lesion/contorno, con margen de
    ## contexto (mismo padding=30 que ya se usaba en el panel completo).
    crop_html = ""
    primera_lesion = overlay_data.lesions[0] if overlay_data.lesions else None
    if primera_lesion and primera_lesion.contours:
        roi, _box = ddsm_overlay.crop_lesion_roi(pil_img, primera_lesion.contours[0], padding=30)
        fig_roi, ax_roi = plt.subplots(figsize=(5, 6.5))
        ax_roi.imshow(np.array(roi.convert("L")), cmap="gray")
        ax_roi.axis("off")
        ax_roi.set_title("Recorte ampliado de la lesion", fontsize=13)
        plt.tight_layout()
        b64_roi = _fig_to_b64(fig_roi)
        crop_html = f'<img src="data:image/png;base64,{b64_roi}" style="width:660px; max-width:100%;">'

    return f'''
    <div style="display:flex; gap:24px; flex-wrap:wrap;">
      <img src="data:image/png;base64,{b64_contorno}" style="width:660px; max-width:100%;">
      {crop_html}
    </div>
    '''


def _reduced_two_line_field_html(campo, linea1_rotulo, linea1_valor, linea2_rotulo, linea2_valor,
                                  color1=PURPLE, color2=GT_TEXT_COLOR):
    ## Formato compartido de dos CAJAS diferenciadas (V1-V4, sobre R2/R3/R4).
    ## Cada linea vive en su propia caja: borde izquierdo grueso del color de
    ## su modulo (morado = lo predicho/generado, modulo de lenguaje, Figura
    ## 1; verde = lo que dice la anotacion, ground truth, Figura 1), fondo
    ## muy tenue del mismo tono (alpha ~8 por ciento via hex de 8 digitos), y
    ## la etiqueta en el color del borde. Una linea horizontal fina separa
    ## las dos cajas. V2: el VALOR de cada campo (no la etiqueta) mantiene
    ## exactamente la misma familia tipografica, cuerpo (20px) y peso
    ## (normal) en ambas cajas -la diferenciacion es solo de color/
    ## contenedor, nunca de tipografia, para no sugerir que uno de los dos
    ## tiene mas autoridad. V4: color2 por defecto es GT_TEXT_COLOR (variante
    ## oscura y legible de GT_BOX_COLOR), no el verde neon original.
    ## white-space:pre-wrap (U1): la linea1 del Bloque 2 puede ser
    ## report_text integro con saltos de linea reales.
    return f'''
    <div style="margin-top:16px;">
      <div style="font-weight:bold; font-size:20px;">{campo}</div>
      <div style="margin-top:6px; padding:8px 12px; border-left:6px solid {color1}; background:{color1}14; white-space:pre-wrap;">
        <span style="font-size:20px; font-weight:normal; color:{color1};">{linea1_rotulo}</span>
        <span style="font-size:20px; font-weight:normal;"> {linea1_valor}</span>
      </div>
      <div style="margin:10px 0; border-top:1px solid #ccc;"></div>
      <div style="padding:8px 12px; border-left:6px solid {color2}; background:{color2}14; white-space:pre-wrap;">
        <span style="font-size:20px; font-weight:normal; color:{color2};">{linea2_rotulo}</span>
        <span style="font-size:20px; font-weight:normal;"> {linea2_valor}</span>
      </div>
    </div>
    '''


def _reduced_block1_html(result):
    ## BLOQUE 1: SALIDA DEL CLASIFICADOR (R2). Ya NO es "coincide"/"no
    ## coincide": mismo formato de dos lineas pegadas que el Bloque 2, con lo
    ## que predijo el modelo y lo que dice la anotacion, sin veredicto -quien
    ## mira el panel compara con sus propios ojos. Densidad (Q2, sigue
    ## vigente): la fila se omite por completo si no hay ground truth de
    ## densidad (los casos DDSM de este demo).
    gt = result["gt"]
    campos = [("patologia", result["pred_pathology"], gt.get("gt_pathology"))]
    campos.append(("BI-RADS", result["birads_level"], gt.get("gt_birads")))
    if gt.get("gt_density_letter") is not None:
        ## result["density_label"] ya trae la letra ACR incluida en el texto
        ## (ej. "tejido mamario heterogeneamente denso (ACR C)"): no se
        ## vuelve a anexar, o queda duplicada como "(ACR C) (ACR C)" (bug
        ## detectado en la verificacion de esta misma ronda).
        campos.append((
            "densidad",
            result["density_label"],
            f"ACR {gt['gt_density_letter']}",
        ))

    campos_html = "".join(
        _reduced_two_line_field_html(
            campo, "lo que predijo el modelo:", valor_predicho,
            "lo que dice la anotacion:", valor_anotacion,
        )
        for campo, valor_predicho, valor_anotacion in campos
    )
    return f'''
    <div style="margin-top:20px; font-size:20px;">
      <div style="font-weight:bold; color:{PURPLE};">BLOQUE 1: SALIDA DEL CLASIFICADOR</div>
      {campos_html}
    </div>
    '''


## S2: patron ESTRICTO del encabezado numerado "N. HALLAZGOS:". No matchea
## "HALLAZGOS:" sin numero (visto en la practica: el reporte del caso 3 no
## trae ninguna seccion numerada) ni "N. **HALLAZGOS:**" con enfasis markdown
## (visto en el reporte oraculo del caso 4) -son casos donde, por diseno, no
## se improvisa un recorte alternativo (S2).
def _reduced_block2_html(result):
    ## BLOQUE 2: DESCRIPCION DEL HALLAZGO (U1-U3). El campo se llama "reporte
    ## generado frente a anotacion" (U3). "lo que dice el reporte" ya NO es
    ## una seccion extraida por encabezado (S2, revertido en U1): es
    ## report_text INTEGRO, textual, con los saltos de linea preservados.
    ## Motivo: el LLM escribe el encabezado de HALLAZGOS de forma
    ## inconsistente entre corridas -con y sin numeracion, con y sin
    ## markdown, con y sin salto de linea tras los dos puntos, verificado en
    ## la conversacion (T1/T2)-, asi que cualquier patron de extraccion es
    ## fragil ante una regeneracion del reporte; mostrar el reporte completo
    ## no depende de ningun patron. "lo que dice la anotacion" no cambia
    ## (U2): descriptores del ground truth traducidos, en una linea.
    hallazgo = result["concordance"]["hallazgo"]
    tipo_anotacion = hallazgo["tipo de lesion"]["anotacion"]
    morf_anotacion = hallazgo["morfologia"]["anotacion"]
    anotacion_valor = f"{tipo_anotacion} / {morf_anotacion}"

    campo_html = _reduced_two_line_field_html(
        "reporte generado frente a anotacion",
        "lo que dice el reporte:", result["report_text"],
        "lo que dice la anotacion:", anotacion_valor,
    )
    return f'''
    <div style="margin-top:20px; font-size:20px;">
      <div style="font-weight:bold; color:{PURPLE};">BLOQUE 2: DESCRIPCION DEL HALLAZGO</div>
      {campo_html}
    </div>
    '''


def render_reduced_panel(result, target=None):
    ## Ensambla el panel de 4 bloques, en el orden de R1-R6: encabezado, las
    ## dos imagenes, Bloque 1 (clasificador), Bloque 2 (descripcion del
    ## hallazgo). Sin frase de veredicto (R1): VERDICT_TEXTS no se usa aqui.
    header_html = _header_reduced_html(result)
    images_html = f'''
    <div style="display:flex; gap:24px; margin-top:16px; flex-wrap:wrap;">
      {_left_image_clean_html(result)}
      {_right_image_reduced_html(result)}
    </div>
    '''
    block1_html = _reduced_block1_html(result)
    block2_html = _reduced_block2_html(result)

    combined_html = header_html + images_html + block1_html + block2_html

    if target is not None:
        target.value = combined_html
    else:
        display(HTML(combined_html))


<h2>Interfaz: 4 botones fijos + boton Limpiar</h2>
<p>Esta es la celda que queda en pantalla durante la grabacion. Cuatro botones fijos,
uno por caso (B8), sustituyen al explorador de archivos (FileChooser): cada boton
pinta el resultado ya precomputado de <code>CASE_RESULTS</code>, sin volver a correr
el pipeline. El boton "Limpiar" borra el panel de resultados sin tocar la cache,
util para dejar la pantalla en blanco entre tomas durante la grabacion.</p>

In [9]:
## Interfaz de 3 botones fijos (P1, ronda de cierre: demo de un minuto). El
## caso 1 (ancla VinDr) NO se borra de CASE_SOURCES/CASE_RESULTS/cache -sigue
## precomputandose arriba y sigue completo en notas_defensa.html-, solo se
## saca de esta lista de botones. Cada boton pinta el PANEL REDUCIDO
## (render_reduced_panel, ver celda anterior), no render_result completo.
##
## Cierre de widgets de una corrida anterior de ESTA MISMA celda, para que
## volver a ejecutarla (por ejemplo con "Run All") no deje botones/paneles
## viejos con manejadores de click activos apuntando a variables globales
## desactualizadas (mismo problema documentado en versiones anteriores de esta
## celda con FileChooser/analyze_button/clear_button).
REDUCED_DEMO_CASE_NUMBERS = [2, 3, 4]

_old_buttons = globals().get("case_buttons")
if _old_buttons:
    for _b in _old_buttons.values():
        try:
            _b.close()
        except Exception:
            pass

for _old_name in ("result_panel", "clear_button"):
    _old = globals().get(_old_name)
    if _old is not None and hasattr(_old, "close"):
        try:
            _old.close()
        except Exception:
            pass

result_panel = widgets.HTML(value="")

case_buttons = {}
for _numero in REDUCED_DEMO_CASE_NUMBERS:
    _source = CASE_SOURCES[_numero]
    _label = f"Caso {_numero}: {_source['categoria']} ({_source['origen']})"
    _btn = widgets.Button(description=_label, layout=widgets.Layout(width="340px"))

    def _make_handler(_n):
        def _handler(_):
            render_reduced_panel(CASE_RESULTS[_n], target=result_panel)
        return _handler

    _btn.on_click(_make_handler(_numero))
    case_buttons[_numero] = _btn

clear_button = widgets.Button(
    description="Limpiar",
    button_style="",
    layout=widgets.Layout(width="140px"),
)


def on_clear_clicked(_):
    result_panel.value = ""


clear_button.on_click(on_clear_clicked)

display(widgets.VBox([
    widgets.HBox([case_buttons[n] for n in sorted(case_buttons)]),
    clear_button,
]))
display(result_panel)


HTML(value='')

<h2>Pantalla de cierre</h2>
<p>Celda final del demo (K1), despues del caso 4. Un boton "Cierre" que
renderiza un panel de una sola pieza, sin imagenes, con el mismo CSS y la
misma tipografia que los paneles de los 4 casos: la frase de cierre del
autor, mas una tabla con las 5 corridas de generacion que corrio este
notebook (los 4 reportes predichos y el oraculo del caso 4), la pagina del
chunk dominante por Shapley y su valor SIN REDONDEAR (los numeros exactos de
J4, no el formato a 3 decimales del resto del panel de auditoria).</p>


In [10]:
## Pantalla de cierre (K1). Panel de una sola pieza, sin imagenes, mismo CSS
## y tipografia que los demas paneles (reusa TEAL/PURPLE de la celda de
## renderizado). No modifica ninguna celda anterior.
##
## CLOSING_PHRASE es un placeholder (el autor no dio el texto final en esta
## ronda): mismo tratamiento que tuvieron los veredictos B12 antes de que el
## autor los fijara. Reemplazar antes de grabar.
CLOSING_PHRASE = (
    "En las dos generaciones donde un fragmento domina claramente la salida, el "
    "caso 1 y la variante con escalares del ground truth del caso 4, es el mismo "
    "fragmento: la figura de calcificaciones agrupadas coarsas y heterogeneas de la "
    "pagina 94 de la referencia BI-RADS. En las otras tres corridas ningun "
    "fragmento domina. Esa misma pagina aparece entre los tres recuperados en tres "
    "de los cuatro casos, incluida una mamografia sin hallazgos. La recuperacion no "
    "depende del caso."
)

## Nota de lectura del valor de Shapley (N3), en cuerpo menor que la frase
## principal. Aclara que "Shapley alto" solo es significativo cuando hay
## dominancia real de un fragmento sobre los demas; en las corridas donde los
## tres chunks tienen valores del mismo orden, el reparto es plano y no hay
## un chunk que domine, aunque el numero en si no sea cero.
CLOSING_TABLE_NOTE = (
    "Shapley alto indica que un fragmento domina la generacion. En las corridas 2, "
    "3 y 4-predicho los valores son del mismo orden entre si: ahi el reparto es "
    "plano y no hay dominancia real."
)


def _closing_screen_html(case_results):
    ##
    ## Arma la tabla de las 5 corridas de generacion a partir de los
    ## resultados ya precomputados (CASE_RESULTS): 4 reportes predichos mas
    ## el oraculo del caso 4. Shapley formateado a 2 decimales (L5, correccion
    ## de una instruccion anterior propia: 16 decimales a 20 px competian con
    ## el argumento del panel en vez de sostenerlo). La precision completa
    ## sigue disponible en outputs/analisis_interno/, no se pierde, solo no
    ## se imprime aqui.
    ##
    runs = []
    for n in sorted(case_results):
        r = case_results[n]
        prov = r["provenance"]
        runs.append((f"Caso {n} (predicho)", prov["source_document"], prov["source_page"], prov["shapley_value"]))
        if "oracle" in r:
            oprov = r["oracle"]["provenance"]
            runs.append((f"Caso {n} (oraculo)", oprov["source_document"], oprov["source_page"], oprov["shapley_value"]))

    rows_html = "".join(
        f"<tr><td style='padding:4px 16px 4px 0;'>{label}</td>"
        f"<td style='padding:4px 16px 4px 0;'>{doc}</td>"
        f"<td style='padding:4px 16px 4px 0;'>{page}</td>"
        f"<td style='padding:4px 0;'>{shapley:.2f}</td></tr>"
        for (label, doc, page, shapley) in runs
    )

    return f'''
    <div style="margin-bottom:16px;">
      <div style="font-size:28px; font-weight:bold; color:{TEAL};">CIERRE</div>
    </div>
    <div style="font-size:20px; margin-top:12px;">
      {CLOSING_PHRASE}
    </div>
    <div style="margin-top:24px; font-size:20px;">
      <div style="font-weight:bold; color:{PURPLE};">LAS 5 CORRIDAS DE GENERACION</div>
      <table style="font-size:20px; border-collapse:collapse; margin-top:8px;">
        <tr style="font-weight:bold;">
          <td style="padding:4px 16px 4px 0;">Corrida</td>
          <td style="padding:4px 16px 4px 0;">Documento</td>
          <td style="padding:4px 16px 4px 0;">Pagina</td>
          <td style="padding:4px 0;">Shapley (2 decimales, precision completa en outputs/analisis_interno/)</td>
        </tr>
        {rows_html}
      </table>
      <div style="margin-top:10px; font-size:14px; color:#666;">
        {CLOSING_TABLE_NOTE}
      </div>
    </div>
    '''


## Cierre de widgets de una corrida anterior de esta celda (mismo patron que
## las demas celdas de interfaz del notebook).
_old_closing_button = globals().get("closing_button")
if _old_closing_button is not None:
    try:
        _old_closing_button.close()
    except Exception:
        pass
_old_closing_panel = globals().get("closing_panel")
if _old_closing_panel is not None:
    try:
        _old_closing_panel.close()
    except Exception:
        pass

closing_panel = widgets.HTML(value="")
closing_button = widgets.Button(description="Cierre", layout=widgets.Layout(width="140px"))


def on_closing_clicked(_):
    closing_panel.value = _closing_screen_html(CASE_RESULTS)


closing_button.on_click(on_closing_clicked)

display(widgets.VBox([closing_button, closing_panel]))
